In [ ]:
!pip install datasets torch numpy

In [ ]:
import torch
import numpy as np
from datasets import load_dataset
import json
import os

In [ ]:
# 1. 파일 경로 설정
file_path = 'train_annotated.json'

# 파일 존재 여부 확인 (업로드 체크)
if not os.path.exists(file_path):
    print(f"오류: '{file_path}' 파일이 현재 경로에 없습니다.")
    print("코랩 좌측의 '폴더' 아이콘을 클릭하고 파일을 업로드한 뒤 다시 실행해 주세요.")
else:
    # 2. 로컬 JSON 파일 로드
    print("1. 로컬 DocRED JSON 데이터를 로드합니다...")
    with open(file_path, 'r', encoding='utf-8') as f:
        dataset = json.load(f)

    print(f"데이터 로드 완료! 총 {len(dataset)}개의 문서가 있습니다.\n")


1. 로컬 DocRED JSON 데이터를 로드합니다...
데이터 로드 완료! 총 3053개의 문서가 있습니다.



In [ ]:
# 3. 원본 JSON 구조에 맞춘 전처리 함수
def build_gain_graphs_raw(vertex_set):
    mentions = []
    entity_of_mention = []

    # 원본 JSON의 이중 리스트 파싱
    for entity_id, entity_group in enumerate(vertex_set):
        for mention in entity_group:
            mentions.append({
                'entity_id': entity_id,
                'name': mention['name'],
                'sent_id': mention['sent_id'],
                'pos': mention['pos']
            })
            entity_of_mention.append(entity_id)

    num_mentions = len(mentions)
    num_entities = len(vertex_set)

    # PyTorch 텐서로 인접 행렬 초기화
    mg_adj = torch.zeros((num_mentions, num_mentions), dtype=torch.float32)
    eg_adj = torch.zeros((num_entities, num_entities), dtype=torch.float32)

    # Mention-level Graph (MG) 간선 연결
    for i in range(num_mentions):
        for j in range(num_mentions):
            if i == j:
                mg_adj[i, j] = 1.0 # Self-loop
                continue

            # 규칙 1: Intra-sentence (같은 문장)
            if mentions[i]['sent_id'] == mentions[j]['sent_id']:
                mg_adj[i, j] = 1.0

            # 규칙 2: Inter-sentence (상호참조)
            elif mentions[i]['entity_id'] == mentions[j]['entity_id']:
                mg_adj[i, j] = 1.0

    # Entity-level Graph (EG) 간선 연결
    for i in range(num_entities):
        for j in range(num_entities):
            if i == j:
                eg_adj[i, j] = 1.0 # Self-loop
                continue

            sents_i = set([m['sent_id'] for m in mentions if m['entity_id'] == i])
            sents_j = set([m['sent_id'] for m in mentions if m['entity_id'] == j])

            # Co-occurrence (동시 등장 문장 확인)
            if len(sents_i.intersection(sents_j)) > 0:
                eg_adj[i, j] = 1.0

    return mg_adj, eg_adj, mentions

In [ ]:
# 4. 파이프라인 실행 및 결과 검증
print("2. 그래프 인접 행렬(Adjacency Matrix) 구축을 시작합니다...\n")

sample_doc = dataset[0]
flat_words = [word for sent in sample_doc['sents'] for word in sent]

print(f"▶ 샘플 문서 제목: {sample_doc.get('title', '제목 없음')}")
print(f"▶ 총 단어 수: {len(flat_words)}개\n")

mg_adj, eg_adj, mention_list = build_gain_graphs_raw(sample_doc['vertexSet'])

print("=== [전처리 파이프라인 결과 검증] ===")
print(f"추출된 총 Mention 개수: {len(mention_list)}")
print(f"통합된 총 Entity 개수: {len(sample_doc['vertexSet'])}")
print("-" * 40)
print(f"[MG 인접 행렬 Tensor 크기]: {mg_adj.shape}")
print(f"[EG 인접 행렬 Tensor 크기]: {eg_adj.shape}")
print(f"EG 행렬 샘플 출력 (상단 5x5):\n{eg_adj[:5, :5]}")

2. 그래프 인접 행렬(Adjacency Matrix) 구축을 시작합니다...

▶ 샘플 문서 제목: AirAsia Zest
▶ 총 단어 수: 197개

=== [전처리 파이프라인 결과 검증] ===
추출된 총 Mention 개수: 27
통합된 총 Entity 개수: 17
----------------------------------------
[MG 인접 행렬 Tensor 크기]: torch.Size([27, 27])
[EG 인접 행렬 Tensor 크기]: torch.Size([17, 17])
EG 행렬 샘플 출력 (상단 5x5):
tensor([[1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.]])


In [ ]:
from transformers import AutoTokenizer
import torch

In [ ]:
print("3. BERT 토크나이저를 로드합니다...")
# GAIN 논문 베이스라인에서 주로 사용하는 bert-base-cased 사용
tokenizer = AutoTokenizer.from_pretrained('bert-base-cased')

3. BERT 토크나이저를 로드합니다...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
def tokenize_and_align(doc, tokenizer):
    """
    문서를 BERT 토큰으로 변환하고, 기존 Mention의 단어 위치(pos)를
    쪼개진 Subword 토큰 위치로 정렬(Alignment)합니다.
    """
    sents = doc['sents']
    vertex_set = doc['vertexSet']

    doc_tokens = []
    word_to_token_map = []

    for sent_idx, sent in enumerate(sents):
        sent_map = []
        for word_idx, word in enumerate(sent):
            # 🌟 에러 해결 핵심: word를 강제로 문자열(str)로 변환!
            subwords = tokenizer.tokenize(str(word))

            start_idx = len(doc_tokens)
            doc_tokens.extend(subwords)
            end_idx = len(doc_tokens)
            sent_map.append((start_idx, end_idx))
        word_to_token_map.append(sent_map)

    mention_token_spans = []

    for entity_id, entity_group in enumerate(vertex_set):
        for mention in entity_group:
            sent_id = mention['sent_id']
            pos_start, pos_end = mention['pos']

            token_start = word_to_token_map[sent_id][pos_start][0]
            token_end = word_to_token_map[sent_id][pos_end - 1][1]

            mention_token_spans.append({
                'entity_id': entity_id,
                'sent_id': sent_id,
                'name': mention['name'],
                'token_span': (token_start, token_end)
            })

    input_ids = tokenizer.convert_tokens_to_ids(doc_tokens)
    input_ids = [tokenizer.cls_token_id] + input_ids + [tokenizer.sep_token_id]

    for span in mention_token_spans:
        span['token_span'] = (span['token_span'][0] + 1, span['token_span'][1] + 1)

    return torch.tensor([input_ids]), mention_token_spans, doc_tokens


In [ ]:
# ==========================================
# 정렬 파이프라인 실행 및 검증
# ==========================================

# 이전 단계에서 썼던 sample_doc(첫 번째 문서) 그대로 사용
input_tensor, aligned_mentions, raw_tokens = tokenize_and_align(sample_doc, tokenizer)

print("\n=== [토큰화 및 정렬 파이프라인 결과 검증] ===")
print(f"▶ 원본 단어 개수: {len(flat_words)}")
print(f"▶ 쪼개진 BERT 토큰 개수: {len(raw_tokens)}")
print(f"▶ 모델 입력 텐서 크기(input_ids): {input_tensor.shape} (CLS, SEP 포함)")
print("-" * 40)

# 정렬이 제대로 되었는지 랜덤으로 2개만 확인
print("[Mention 위치 정렬 테스트]")
for i in range(min(2, len(aligned_mentions))):
    span = aligned_mentions[i]['token_span']
    name = aligned_mentions[i]['name']

    # input_tensor에서 해당 위치의 토큰 ID를 가져와 텍스트로 복원
    extracted_tokens = tokenizer.convert_ids_to_tokens(input_tensor[0][span[0]:span[1]])
    extracted_text = tokenizer.convert_tokens_to_string(extracted_tokens)

    print(f"원본 개체명: {name:15} | 정렬된 토큰 위치: {span} | 해당 위치 토큰 복원: {extracted_text}")


=== [토큰화 및 정렬 파이프라인 결과 검증] ===
▶ 원본 단어 개수: 197
▶ 쪼개진 BERT 토큰 개수: 219
▶ 모델 입력 텐서 크기(input_ids): torch.Size([1, 221]) (CLS, SEP 포함)
----------------------------------------
[Mention 위치 정렬 테스트]
원본 개체명: Zest Airways, Inc. | 정렬된 토큰 위치: (1, 7) | 해당 위치 토큰 복원: Zest Airways, Inc.
원본 개체명: Asian Spirit and Zest Air | 정렬된 토큰 위치: (16, 22) | 해당 위치 토큰 복원: Asian Spirit and Zest Air


In [ ]:
import torch
import torch.nn as nn
from transformers import AutoModel

# 1. 그래프 정규화 도우미 함수 (대칭 정규화)
def normalize_adj(adj):
    """
    인접 행렬 A를 D^{-1/2} * A * D^{-1/2} 형태로 정규화합니다.
    (허브 노드가 가중치를 독식하는 것을 방지)
    """
    # 노드별 차수(Degree) 계산 (행 단위 합)
    rowsum = adj.sum(dim=1)
    # 0으로 나누는 것을 방지하기 위해 아주 작은 값(1e-9) 추가 후 역수 제곱근
    d_inv_sqrt = torch.pow(rowsum + 1e-9, -0.5)
    # 대각 행렬로 변환
    d_mat_inv_sqrt = torch.diag(d_inv_sqrt)
    # 정규화된 행렬 반환
    normalized_adj = torch.matmul(torch.matmul(d_mat_inv_sqrt, adj), d_mat_inv_sqrt)
    return normalized_adj

# 2. GCN 기본 레이어
class GCNLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super(GCNLayer, self).__init__()
        self.linear = nn.Linear(in_dim, out_dim)
        self.relu = nn.ReLU()

    def forward(self, node_features, adj):
        # H^(l+1) = ReLU(A * H^l * W)
        out = torch.matmul(adj, node_features) # 이웃 정보 수집
        out = self.linear(out)                 # 가중치 곱
        return self.relu(out)                  # 비선형 활성화

# 3. GAIN MVP 메인 모델
class GAIN_MVP(nn.Module):
    def __init__(self, plm_name='bert-base-cased', hidden_dim=768):
        super(GAIN_MVP, self).__init__()
        # BERT 인코더 로드
        self.encoder = AutoModel.from_pretrained(plm_name)

        # MG와 EG를 위한 GCN 레이어
        self.mg_gcn = GCNLayer(hidden_dim, hidden_dim)
        self.eg_gcn = GCNLayer(hidden_dim, hidden_dim)

    def forward(self, input_ids, mg_adj, eg_adj, aligned_mentions, num_entities):
        # 1. 언어 모델을 통한 전체 문맥 인코딩
        # BERT 출력의 last_hidden_state (배치, 시퀀스길이, 768)
        outputs = self.encoder(input_ids=input_ids)
        context_out = outputs.last_hidden_state[0] # MVP이므로 배치 사이즈 1 가정

        # 2. Mention 노드 벡터 추출 (좌표를 이용한 Pooling)
        mention_features = []
        for m in aligned_mentions:
            start, end = m['token_span']
            # 시작 위치부터 끝 위치까지의 토큰 벡터들을 평균(Mean) 내어 Mention 벡터 구성
            span_vec = context_out[start:end].mean(dim=0)
            mention_features.append(span_vec)

        # (Mention 개수, 768) 크기의 텐서로 변환
        h_mg_0 = torch.stack(mention_features)

        # 3. Mention-level Graph 연산
        norm_mg_adj = normalize_adj(mg_adj)
        h_mg_1 = self.mg_gcn(h_mg_0, norm_mg_adj)

        # 4. Mention -> Entity 노드 병합 (Aggregation)
        entity_features = []
        for entity_id in range(num_entities):
            # 현재 Entity ID에 속하는 Mention 벡터들의 인덱스 찾기
            m_indices = [i for i, m in enumerate(aligned_mentions) if m['entity_id'] == entity_id]
            # 해당 벡터들을 가져와서 평균 내기
            if len(m_indices) > 0:
                ent_vec = h_mg_1[m_indices].mean(dim=0)
            else:
                ent_vec = torch.zeros(768) # 만약 비어있다면 0벡터
            entity_features.append(ent_vec)

        h_eg_0 = torch.stack(entity_features) # (Entity 개수, 768)

        # 5. Entity-level Graph 연산 (Multi-hop Reasoning)
        norm_eg_adj = normalize_adj(eg_adj)
        h_eg_1 = self.eg_gcn(h_eg_0, norm_eg_adj)

        return h_mg_1, h_eg_1

# ==========================================
# GCN 추론 파이프라인 실행 및 검증
# ==========================================

print("4. GAIN 모델을 초기화하고 연산을 수행합니다 (초기화에 수 초 소요)...")
# 이전 단계에서 만든 변수들(input_tensor, mg_adj, eg_adj, aligned_mentions) 사용
num_entities = len(sample_doc['vertexSet'])
model = GAIN_MVP()

# 기울기 계산 비활성화 (순전파 테스트만 진행)
with torch.no_grad():
    mg_out, eg_out = model(
        input_ids=input_tensor,
        mg_adj=mg_adj,
        eg_adj=eg_adj,
        aligned_mentions=aligned_mentions,
        num_entities=num_entities
    )

print("\n=== [GCN 연산 파이프라인 결과 검증] ===")
print("문서의 비정형 텍스트가 지식 그래프 벡터로 완벽히 압축되었습니다!")
print("-" * 50)
print(f"▶ 초기 Mention 개수: {mg_adj.shape[0]}개")
print(f"▶ MG GCN 통과 후 Mention 텐서 크기: {mg_out.shape} (각 단어가 문서 문맥을 흡수함)")
print("-" * 50)
print(f"▶ 병합된 Entity 개수: {num_entities}개")
print(f"▶ EG GCN 통과 후 최종 Entity 텐서 크기: {eg_out.shape} (최종 다중 도약 추론 결과)")

4. GAIN 모델을 초기화하고 연산을 수행합니다 (초기화에 수 초 소요)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



=== [GCN 연산 파이프라인 결과 검증] ===
문서의 비정형 텍스트가 지식 그래프 벡터로 완벽히 압축되었습니다!
--------------------------------------------------
▶ 초기 Mention 개수: 27개
▶ MG GCN 통과 후 Mention 텐서 크기: torch.Size([27, 768]) (각 단어가 문서 문맥을 흡수함)
--------------------------------------------------
▶ 병합된 Entity 개수: 17개
▶ EG GCN 통과 후 최종 Entity 텐서 크기: torch.Size([17, 768]) (최종 다중 도약 추론 결과)


In [ ]:
import torch
import torch.nn as nn
import itertools

# 1. 관계 분류기 모듈 정의
class RelationClassifier(nn.Module):
    def __init__(self, hidden_dim=768, num_relations=97):
        super(RelationClassifier, self).__init__()
        # 두 객체 벡터를 이어 붙여서(Concat: 768 * 2) 관계를 추론하는 Linear 모델
        self.bilinear = nn.Linear(hidden_dim * 2, num_relations)
        # 다중 라벨 분류를 위한 독립적 시그모이드
        self.sigmoid = nn.Sigmoid()

    def forward(self, entity_embeddings):
        num_entities = entity_embeddings.size(0)

        # 결과를 담을 리스트
        pair_results = []

        # 1. 모든 가능한 객체 쌍(Pair) 생성 (자기 자신 제외, A->B 와 B->A는 다른 방향임)
        for head_idx, tail_idx in itertools.permutations(range(num_entities), 2):
            # 주체(Head)와 대상(Tail) 벡터 추출
            head_vec = entity_embeddings[head_idx]
            tail_vec = entity_embeddings[tail_idx]

            # 두 벡터를 이어 붙임 (Concatenation)
            pair_vec = torch.cat([head_vec, tail_vec], dim=0)

            # 2. 로짓(Logit) 점수 계산 및 시그모이드 확률 변환
            logits = self.bilinear(pair_vec)
            probs = self.sigmoid(logits) # (97,) 크기의 확률 벡터

            pair_results.append({
                'head_idx': head_idx,
                'tail_idx': tail_idx,
                'probs': probs
            })

        return pair_results

# ==========================================
# 관계 추출 파이프라인 최종 실행
# ==========================================

print("5. 최종 관계 분류기를 통과시켜 지식(Knowledge)을 추출합니다...\n")

# DocRED의 관계 라벨 예시 (출력을 보기 좋게 하기 위한 임시 딕셔너리)
mock_relation_names = {
    0: "NA (관계 없음)",
    17: "국가 (P17)",
    112: "설립자 (P112)",
    159: "본사 위치 (P159)"
}

# 분류기 초기화 (MVP 모델이므로 가중치는 랜덤 초기화 상태)
classifier = RelationClassifier()

# 이전 단계의 eg_out (최종 Entity 텐서) 사용
with torch.no_grad():
    predictions = classifier(eg_out)

print("=== [최종 지식 그래프 추출 결과] ===")
print(f"▶ 생성된 총 객체 쌍(Pairs)의 수: {len(predictions)}개 ( N * (N-1) )")
print("-" * 50)

# 추출 결과 확인 (임계값 0.7 적용)
threshold = 0.7
extracted_triples = 0

# 상위 3개의 쌍만 샘플로 확인
for idx, pred in enumerate(predictions[:3]):
    h_idx = pred['head_idx']
    t_idx = pred['tail_idx']
    probs = pred['probs']

    # 해당 Entity의 원래 이름 찾기 (첫 번째 Mention 기준)
    h_name = next(m['name'] for m in aligned_mentions if m['entity_id'] == h_idx)
    t_name = next(m['name'] for m in aligned_mentions if m['entity_id'] == t_idx)

    # 확률이 Threshold를 넘는 관계 번호 찾기
    predicted_rels = (probs > threshold).nonzero(as_tuple=True)[0].tolist()

    print(f"[객체 쌍 {idx+1}] 주체: '{h_name}' ---> 대상: '{t_name}'")

    if not predicted_rels:
        print("  => 발견된 관계 없음 (모든 확률이 임계값 미달)")
    else:
        for rel_id in predicted_rels:
            # 임시 딕셔너리에 이름이 없으면 그냥 ID 출력
            rel_name = mock_relation_names.get(rel_id, f"관계ID_{rel_id}")
            rel_prob = probs[rel_id].item() * 100
            print(f"  => 발견된 관계: {rel_name} (확률: {rel_prob:.1f}%)")
            extracted_triples += 1
    print("- " * 20)

print(f"\n🎉 파이프라인 완료! (MVP 특성상 모델 학습 전이라 결과는 무작위로 추출됩니다.)")

5. 최종 관계 분류기를 통과시켜 지식(Knowledge)을 추출합니다...

=== [최종 지식 그래프 추출 결과] ===
▶ 생성된 총 객체 쌍(Pairs)의 수: 272개 ( N * (N-1) )
--------------------------------------------------
[객체 쌍 1] 주체: 'Zest Airways, Inc.' ---> 대상: 'Ninoy Aquino International Airport'
  => 발견된 관계 없음 (모든 확률이 임계값 미달)
- - - - - - - - - - - - - - - - - - - - 
[객체 쌍 2] 주체: 'Zest Airways, Inc.' ---> 대상: 'Pasay City'
  => 발견된 관계 없음 (모든 확률이 임계값 미달)
- - - - - - - - - - - - - - - - - - - - 
[객체 쌍 3] 주체: 'Zest Airways, Inc.' ---> 대상: 'Metro Manila'
  => 발견된 관계 없음 (모든 확률이 임계값 미달)
- - - - - - - - - - - - - - - - - - - - 

🎉 파이프라인 완료! (MVP 특성상 모델 학습 전이라 결과는 무작위로 추출됩니다.)


In [ ]:
import torch.optim as optim
import torch.nn as nn

# 1. 옵티마이저와 손실 함수 정의
# BERT 모델의 가중치와 GCN, Classifier의 가중치를 모두 학습 대상으로 등록 (학습률은 1e-5 등 작은 값 사용)
optimizer = optim.AdamW(list(model.parameters()) + list(classifier.parameters()), lr=1e-5)

# BCELoss: 분류기의 Sigmoid 출력값과 0, 1로 이루어진 정답지 사이의 오차를 계산
criterion = nn.BCELoss()

def train_step(model, classifier, optimizer, criterion, input_tensor, mg_adj, eg_adj, aligned_mentions, num_entities, sample_doc):
    """
    하나의 문서(Document)에 대해 1회 학습(Forward -> Loss -> Backward -> Update)을 수행합니다.
    """
    # 모델을 학습 모드로 전환 (Dropout 등이 활성화됨)
    model.train()
    classifier.train()

    # 1단계: 누적된 기울기(Gradient) 초기화
    optimizer.zero_grad()

    # 2단계: 순전파 (Forward Pass) - 모델에 데이터를 넣고 예측값 뽑기
    mg_out, eg_out = model(
        input_ids=input_tensor,
        mg_adj=mg_adj,
        eg_adj=eg_adj,
        aligned_mentions=aligned_mentions,
        num_entities=num_entities
    )
    predictions = classifier(eg_out) # [{'head_idx': 0, 'tail_idx': 1, 'probs': tensor([...])}, ...]

    # 3단계: 정답지(Target) 텐서 만들기
    # 예측한 객체 쌍의 개수만큼 정답 텐서 초기화 (모두 0으로)
    num_pairs = len(predictions)
    targets = torch.zeros((num_pairs, 97), dtype=torch.float32)

    # DocRED의 실제 정답(labels) 파싱
    # 예: [{'h': 0, 't': 1, 'r': 'P159'}, ...]
    true_labels = sample_doc.get('labels', [])

    # P-ID를 인덱스 번호로 바꾸는 매핑이 필요하지만, 여기서는 임시로 P159를 인덱스 159 대신
    # mock_relation_names에 있던 임의의 번호(예: 159 -> 인덱스 3)로 들어간다고 가정하고 로직만 구현합니다.
    # (실제 구현 시에는 rel2id.json 이라는 딕셔너리를 로드해서 'P159' -> 43 처럼 변환해야 합니다.)

    # 정답 텐서에 1.0(정답) 표시
    for pred_idx, pred in enumerate(predictions):
        h_idx = pred['head_idx']
        t_idx = pred['tail_idx']

        # 현재 예측 중인 쌍(h_idx, t_idx)이 실제 정답지에 있는지 확인
        for true_label in true_labels:
            if true_label['h'] == h_idx and true_label['t'] == t_idx:
                # 임시: 원래는 rel2id[true_label['r']] 로 인덱스를 가져와야 함.
                # 여기서는 작동 확인을 위해 P159면 159번 인덱스에 1을 넣는다고 가정 (실제 97개 클래스 매핑 필요)
                # target_rel_idx = rel2id[true_label['r']]
                # targets[pred_idx, target_rel_idx] = 1.0
                pass

    # 모델의 예측 확률 텐서들을 하나로 모으기 (num_pairs, 97)
    predicted_probs = torch.stack([pred['probs'] for pred in predictions])

    # 4단계: 오차 계산 (Loss Calculation)
    # 모델의 예측 확률(predicted_probs)과 실제 정답(targets) 비교
    loss = criterion(predicted_probs, targets)

    # 5단계: 역전파 (Backward Pass) - 오차를 바탕으로 각 가중치의 기울기 계산
    loss.backward()

    # 6단계: 가중치 업데이트 (Optimization Step)
    optimizer.step()

    return loss.item()

# ==========================================
# 학습 루프 실행 테스트
# ==========================================
print("🚀 학습(Training) 단계를 시작합니다...")

# 학습 전 초기 Loss 확인
initial_loss = train_step(model, classifier, optimizer, criterion, input_tensor, mg_adj, eg_adj, aligned_mentions, num_entities, sample_doc)
print(f"▶ 1회차 학습 완료! 발생한 오차(Loss): {initial_loss:.4f}")

# 반복 학습 시뮬레이션
print("▶ 동일한 문서로 5회 반복 학습을 진행하며 Loss가 떨어지는지 최적화 과정을 확인합니다.")
for epoch in range(5):
    current_loss = train_step(model, classifier, optimizer, criterion, input_tensor, mg_adj, eg_adj, aligned_mentions, num_entities, sample_doc)
    print(f"  - Epoch {epoch+1} Loss: {current_loss:.4f}")

🚀 학습(Training) 단계를 시작합니다...
▶ 1회차 학습 완료! 발생한 오차(Loss): 0.6966
▶ 동일한 문서로 5회 반복 학습을 진행하며 Loss가 떨어지는지 최적화 과정을 확인합니다.
  - Epoch 1 Loss: 0.6943
  - Epoch 2 Loss: 0.6922
  - Epoch 3 Loss: 0.6902
  - Epoch 4 Loss: 0.6882
  - Epoch 5 Loss: 0.6863


In [ ]:

def evaluate(model, classifier, dev_dataset, tokenizer):
    """
    검증 데이터셋을 통해 모델의 현재 성능(F1 Score 등)을 측정합니다.
    """
    model.eval()       # 평가 모드 전환 (Dropout 비활성화)
    classifier.eval()

    all_predictions = []
    all_targets = []

    # 🌟 기울기 계산을 끔 (메모리 절약 및 업데이트 방지)
    with torch.no_grad():
        for doc in dev_dataset:
            # 1. 전처리 (이전에 만든 함수들 활용)
            mg_adj, eg_adj, _ = build_gain_graphs_raw(doc['vertexSet'])
            input_tensor, aligned_mentions, _ = tokenize_and_align(doc, tokenizer)
            num_entities = len(doc['vertexSet'])

            # 2. 순전파 (Forward Pass)
            mg_out, eg_out = model(input_tensor, mg_adj, eg_adj, aligned_mentions, num_entities)
            pair_probs = classifier(eg_out)

            # 3. 임계값(0.7) 적용하여 관계 추출
            # (이 결과를 실제 labels와 비교하여 F1 Score 계산 로직 실행)
            # ...

    print("✅ 검증 완료: 현재 F1 Score: 58.2% (예시)")

In [ ]:
import os
import json
import random
import torch
import torch.optim as optim
import torch.nn as nn

# 1. 단일 원본 파일 로드 및 데이터 분할 (Train-Test Split)
file_path = 'train_annotated.json'

if not os.path.exists(file_path):
    raise FileNotFoundError(f"코랩 폴더에 '{file_path}' 파일이 없습니다. 업로드해 주세요.")

with open(file_path, 'r', encoding='utf-8') as f:
    full_dataset = json.load(f)

print(f"원본 데이터 로드 완료! 총 {len(full_dataset)}개의 문서가 있습니다.")

# 재현성(매번 똑같이 분할되도록)을 위해 시드 고정
random.seed(42)
# 데이터 무작위로 섞기
random.shuffle(full_dataset)

# 💡 빠른 MVP 테스트를 위해 전체 중 10개 문서만 샘플링해서 8:2로 나눕니다.
# (전체 데이터로 진짜 학습을 돌릴 때는 아래 slicing 10을 지워주세요: full_dataset[:])
sample_dataset = full_dataset[:10]

# 80%는 학습용, 20%는 검증용으로 인덱스 계산
split_idx = int(len(sample_dataset) * 0.8)

train_dataset = sample_dataset[:split_idx]
dev_dataset = sample_dataset[split_idx:]

print(f"데이터 8:2 분할 완료! 👉 학습(Train): {len(train_dataset)}개 | 검증(Validation): {len(dev_dataset)}개\n")

# 2. 모델 및 학습 도구 초기화
model = GAIN_MVP()
classifier = RelationClassifier()

optimizer = optim.AdamW(list(model.parameters()) + list(classifier.parameters()), lr=1e-5)
criterion = nn.BCELoss()
threshold = 0.7
mock_rel2id = {'P159': 1, 'P17': 2, 'P112': 3}

# 3. 메인 파이프라인 실행 루프 (train_epoch, evaluate 함수는 이전과 동일)
epochs = 100

print("🚀 데이터 분할 기반 파이프라인(학습-검증)을 시작합니다...\n")

for epoch in range(epochs):
    # Train
    model.train()
    classifier.train()
    total_loss = 0.0

    for doc in train_dataset:
        optimizer.zero_grad()
        mg_adj, eg_adj, _ = build_gain_graphs_raw(doc['vertexSet'])
        input_tensor, aligned_mentions, _ = tokenize_and_align(doc)
        num_entities = len(doc['vertexSet'])

        mg_out, eg_out = model(input_tensor, mg_adj, eg_adj, aligned_mentions, num_entities)
        predictions = classifier(eg_out)

        num_pairs = len(predictions)
        targets = torch.zeros((num_pairs, 97), dtype=torch.float32)
        true_labels = doc.get('labels', [])

        for pred_idx, pred in enumerate(predictions):
            for true_label in true_labels:
                if true_label['h'] == pred['head_idx'] and true_label['t'] == pred['tail_idx']:
                    rel_idx = mock_rel2id.get(true_label['r'], 0)
                    targets[pred_idx, rel_idx] = 1.0

        predicted_probs = torch.stack([pred['probs'] for pred in predictions])

        loss = criterion(predicted_probs, targets)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    avg_train_loss = total_loss / len(train_dataset)

    # Validation
    precision, recall, f1_score = evaluate(model, classifier, dev_dataset, threshold)

    # 결과 출력
    print(f"[Epoch {epoch+1}/{epochs}]")
    print(f"  - Train Loss: {avg_train_loss:.4f}")
    print(f"  - Validation: Precision {precision:.4f} | Recall {recall:.4f} | ✨ F1 Score {f1_score:.4f}")
    print("-" * 50)

원본 데이터 로드 완료! 총 3053개의 문서가 있습니다.
데이터 8:2 분할 완료! 👉 학습(Train): 8개 | 검증(Validation): 2개



Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


🚀 데이터 분할 기반 파이프라인(학습-검증)을 시작합니다...

[Epoch 1/100]
  - Train Loss: 0.6893
  - Validation: Precision 0.0000 | Recall 0.0000 | ✨ F1 Score 0.0000
--------------------------------------------------
[Epoch 2/100]
  - Train Loss: 0.6765
  - Validation: Precision 0.0000 | Recall 0.0000 | ✨ F1 Score 0.0000
--------------------------------------------------
[Epoch 3/100]
  - Train Loss: 0.6583
  - Validation: Precision 0.0000 | Recall 0.0000 | ✨ F1 Score 0.0000
--------------------------------------------------
[Epoch 4/100]
  - Train Loss: 0.6267
  - Validation: Precision 0.0000 | Recall 0.0000 | ✨ F1 Score 0.0000
--------------------------------------------------
[Epoch 5/100]
  - Train Loss: 0.5881
  - Validation: Precision 0.0000 | Recall 0.0000 | ✨ F1 Score 0.0000
--------------------------------------------------
[Epoch 6/100]
  - Train Loss: 0.5487
  - Validation: Precision 0.0000 | Recall 0.0000 | ✨ F1 Score 0.0000
--------------------------------------------------
[Epoch 7/100]
  - Trai

In [ ]:
# 1. 토크나이저를 '확실하게' 다시 초기화 (숫자로 덮어씌워진 것 초기화)
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained('bert-base-cased')

# 2. 파라미터에서 tokenizer를 아예 삭제 (순서 꼬임 원천 차단)
def tokenize_and_align(doc):
    sents = doc['sents']
    vertex_set = doc['vertexSet']

    doc_tokens = []
    word_to_token_map = []

    for sent_idx, sent in enumerate(sents):
        sent_map = []
        for word_idx, word in enumerate(sent):
            # 전역 변수 tokenizer를 바로 사용
            subwords = tokenizer.tokenize(str(word))
            start_idx = len(doc_tokens)
            doc_tokens.extend(subwords)
            end_idx = len(doc_tokens)
            sent_map.append((start_idx, end_idx))
        word_to_token_map.append(sent_map)

    mention_token_spans = []
    for entity_id, entity_group in enumerate(vertex_set):
        for mention in entity_group:
            sent_id = mention['sent_id']
            pos_start, pos_end = mention['pos']

            token_start = word_to_token_map[sent_id][pos_start][0]
            token_end = word_to_token_map[sent_id][pos_end - 1][1]

            mention_token_spans.append({
                'entity_id': entity_id,
                'sent_id': sent_id,
                'name': mention['name'],
                'token_span': (token_start, token_end)
            })

    input_ids = tokenizer.convert_tokens_to_ids(doc_tokens)
    input_ids = [tokenizer.cls_token_id] + input_ids + [tokenizer.sep_token_id]

    for span in mention_token_spans:
        span['token_span'] = (span['token_span'][0] + 1, span['token_span'][1] + 1)

    return torch.tensor([input_ids]), mention_token_spans, doc_tokens


# 3. 검증 함수에서도 파라미터에서 tokenizer 삭제
def evaluate(model, classifier, dataset, threshold):
    model.eval()
    classifier.eval()
    tp, fp, fn = 0, 0, 0
    with torch.no_grad():
        for doc in dataset:
            mg_adj, eg_adj, _ = build_gain_graphs_raw(doc['vertexSet'])

            # 파라미터 없이 바로 호출!
            input_tensor, aligned_mentions, _ = tokenize_and_align(doc)

            mg_out, eg_out = model(input_tensor, mg_adj, eg_adj, aligned_mentions, len(doc['vertexSet']))
            predictions = classifier(eg_out)
            true_labels = doc.get('labels', [])

            for pred in predictions:
                h_idx, t_idx, probs = pred['head_idx'], pred['tail_idx'], pred['probs']
                predicted_rels = set((probs >= threshold).nonzero(as_tuple=True)[0].tolist())
                actual_rels = set([mock_rel2id.get(tl['r'], 0) for tl in true_labels if tl['h'] == h_idx and tl['t'] == t_idx])

                tp += len(predicted_rels.intersection(actual_rels))
                fp += len(predicted_rels - actual_rels)
                fn += len(actual_rels - predicted_rels)

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
    return precision, recall, f1